# Rotating Brain Animations: Sensitivity, Probes, Parcels and DOT Images

This notebook turns standard cedalion results into looping, rotating GIF animations,
e.g. for talks, posters or websites. Every animation uses the same recipe: a
PyVista scene is rendered off-screen while the camera orbits the head around its
vertical (z) axis from a slightly elevated angle, so that one loop shows the head or
brain from all sides except from below.

**Animations produced in this notebook** (all on the Colin27 standard head model):

1. Whole-head **sensitivity profile** of the NinjaNIRS 56x144 cap
2. The **NinjaNIRS probe** on the head, with a semi-transparent scalp above the brain
3. The **Schaefer parcellation** on (a) the pial and (b) the inflated cortex surface
4. The **block-averaged DOT image reconstruction** of a finger-tapping experiment,
   (a) as the multi-view animation from
   [40_image_reconstruction.ipynb](40_image_reconstruction.ipynb) and (b) on a
   rotating brain, where time and camera angle advance together

All inputs (head model, probe geometry, precomputed sensitivities, finger-tapping
recording) are downloaded automatically via `cedalion.data`, so no GPU or photon
simulation is needed.

**Related notebooks:**
- [43a_head_models_overview.ipynb](43a_head_models_overview.ipynb): head models,
  parcels and the inflated surface
- [46_precompute_fluence.ipynb](46_precompute_fluence.ipynb): how the precomputed
  sensitivities were generated
- [40_image_reconstruction.ipynb](40_image_reconstruction.ipynb): the full DOT image
  reconstruction pipeline used in section 4

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv
import xarray as xr
from IPython.display import Image as IPImage
from IPython.display import display
from PIL import Image
from scipy.spatial import cKDTree

import cedalion
import cedalion.data
import cedalion.dataclasses as cdc
import cedalion.dot as dot
import cedalion.nirs
import cedalion.sigproc.motion as motion
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx
from cedalion import units

# All scenes are rendered off-screen, so no interactive backend is needed.
pv.set_jupyter_backend("static")
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

xr.set_options(display_expand_data=False);

## Configuration

All animations share these settings. The defaults produce 800x800 px GIFs that show
one full 360° turn in 5 s and then loop.

- `N_FRAMES` controls smoothness *and* file size. With 100 frames, each GIF is
  roughly 6-17 MB. Use e.g. 50 frames and a 500x500 px window for web-sized files.
- `ELEVATION_DEG` is the camera angle above the horizontal plane. Around 20-30°
  shows the top of the brain well without hiding the temporal lobes.
- `MAKE` selects which animations to render. Section 4 runs a short preprocessing
  and image reconstruction pipeline and takes the longest.

In [ ]:
OUTPUT_DIR = Path("rotating_brain_gifs")  # where the GIFs are written

DURATION_S = 5.0          # length of one loop
N_FRAMES = 100            # frames per loop (100 frames / 5 s = 20 fps)
ELEVATION_DEG = 25        # camera elevation above the horizontal plane
WINDOW_SIZE = (800, 800)  # rendered image size in pixels

HEAD_MODEL = "colin27"    # "colin27" or "icbm152"
PROBE = "ninja_cap_56x144"

MAKE = {
    "sensitivity": True,       # 1
    "probe": True,             # 2
    "parcels_pial": True,      # 3a
    "parcels_inflated": True,  # 3b
    "recon_multiview": True,   # 4a
    "recon_rotating": True,    # 4b
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions

Three small helpers do the work that is common to all animations:

- `orbit_camera` places the camera on a circle around a focal point. The azimuth
  `theta` is measured from the +x axis (the subject's right) towards +y (anterior),
  so `theta = 0` looks at the right hemisphere and `theta = pi/2` at the face.
- `render_orbit` renders one frame per camera position. An optional
  `update(i_frame)` callback can change the scene between frames. Section 4 uses it
  to advance the time series while the camera rotates.
- `save_gif` writes the frames as a looping GIF. GIF frame durations are stored in
  units of 10 ms, so the durations are spread across frames to add up to exactly
  `DURATION_S`. GIFs store at most 256 colours, so `quantize_frames` builds one
  palette from frames sampled over the *whole* turn and maps every pixel to its
  nearest palette colour. A palette taken from a single frame would lack colours
  that only appear at other viewing angles, and those would be replaced by wrong
  colours (visible as speckles). Sharing one palette also avoids colour flicker
  between frames.

In [ ]:
def orbit_camera(plotter, center, distance, theta, elevation_deg=ELEVATION_DEG):
    # Place the camera on an orbit around `center`, looking at it with z up.
    el = np.deg2rad(elevation_deg)
    center = np.asarray(center, dtype=float)
    plotter.camera.position = center + distance * np.array(
        [np.cos(el) * np.cos(theta), np.cos(el) * np.sin(theta), np.sin(el)]
    )
    plotter.camera.focal_point = center
    plotter.camera.up = (0, 0, 1)


def render_orbit(plotter, center, distance, n_frames=N_FRAMES, theta0=np.pi / 2,
                 update=None):
    # Render one full turn around the z axis and return the frames as PIL images.
    thetas = theta0 + np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    frames = []
    for i_frame, theta in enumerate(thetas):
        if update is not None:
            update(i_frame)
        orbit_camera(plotter, center, distance, theta)
        plotter.render()
        frames.append(Image.fromarray(plotter.screenshot(return_img=True)))
    plotter.close()
    return frames


def quantize_frames(frames, n_colors=256, n_sample=12):
    # Map RGB frames onto one shared palette sampled from the whole loop.
    idx = np.linspace(0, len(frames) - 1, min(n_sample, len(frames)), dtype=int)
    sample = np.concatenate([np.asarray(frames[i].convert("RGB")) for i in idx], axis=1)
    pal_img = Image.fromarray(sample).quantize(
        colors=n_colors, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE
    )
    palette = np.asarray(pal_img.getpalette()[: 3 * n_colors], dtype=np.uint8)
    palette = palette.reshape(-1, 3)
    tree = cKDTree(palette.astype(float))

    quantized = []
    for f in frames:
        rgb = np.asarray(f.convert("RGB")).astype(np.int32)
        # exact nearest-colour lookup, done once per unique colour for speed
        key = (rgb[..., 0] << 16) | (rgb[..., 1] << 8) | rgb[..., 2]
        uniq, inv = np.unique(key.ravel(), return_inverse=True)
        uniq_rgb = np.stack([(uniq >> 16) & 255, (uniq >> 8) & 255, uniq & 255], 1)
        _, nearest = tree.query(uniq_rgb.astype(float))
        img = Image.fromarray(nearest[inv].astype(np.uint8).reshape(rgb.shape[:2]), "P")
        img.putpalette(palette.ravel().tolist())
        quantized.append(img)
    return quantized


def save_gif(frames, fname, duration_s=DURATION_S):
    # Save frames as a looping GIF whose frame durations sum to duration_s.
    n = len(frames)
    total_cs = int(round(duration_s * 100))  # GIF durations are in 1/100 s
    base, extra = divmod(total_cs, n)
    longer = set(np.linspace(0, n - 1, extra, dtype=int)) if extra else set()
    durations = [10 * (base + (i in longer)) for i in range(n)]

    frames = quantize_frames(frames)
    frames[0].save(fname, save_all=True, append_images=frames[1:],
                   duration=durations, loop=0, disposal=1)
    print(f"{fname}: {n} frames, {sum(durations) / 1000:.2f} s")


def load_gif_frames(fname):
    # Read all frames of an existing GIF, e.g. to retime it with save_gif.
    im = Image.open(fname)
    frames = []
    for i in range(im.n_frames):
        im.seek(i)
        frames.append(im.convert("RGB"))
    return frames


def new_plotter():
    plotter = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
    plotter.enable_anti_aliasing("ssaa")
    return plotter


def show_gif(fname):
    display(IPImage(data=open(fname, "rb").read(), format="png"))

## Load the head model

`dot.get_standard_headmodel` returns a `TwoSurfaceHeadModel` in voxel space
(`crs="ijk"`). For the animations we transform it into the atlas' scanner space with
`t_ijk2ras`. There, coordinates are in mm and the z axis points in the superior
direction, which makes it the natural rotation axis.

The brain surface carries a `parcel` label for each vertex (Schaefer atlas), and the
inflated cortex surface shares the same vertex order. Any per-vertex quantity can
therefore be shown on either surface.

In [ ]:
head_ijk = dot.get_standard_headmodel(HEAD_MODEL)
head = head_ijk.apply_transform(head_ijk.t_ijk2ras)
infl = dot.get_inflated_cortex_surface(HEAD_MODEL)


def surface_center(surface):
    # Center of the surface's bounding box, used as camera focal point.
    return np.asarray(surface.mesh.bounds).mean(axis=0)


print("CRS:", head.crs, "| brain vertices:", head.brain.nvertices)

## 1. Whole-head sensitivity profile

`cedalion.data.get_precomputed_sensitivity` loads a sensitivity matrix `Adot` with
dimensions `(channel, vertex, wavelength)`. It was computed with Monte Carlo photon
simulations (MCX) for the NinjaNIRS cap registered to the head model. Each entry
describes how strongly an absorption change at one vertex affects the optical
density of one channel.

To summarize the whole-head coverage, we sum over all channels, keep only brain
vertices (`is_brain`), and show the result on a log10 scale. The colour range of
-2 to 1 resolves the drop-off into sulci and towards the inferior brain. Wider
ranges such as -3 to 0 saturate most of the cortex for this dense cap.

In [ ]:
SENS_WAVELENGTH = 760.0  # nm
SENS_CLIM = (-2, 1)      # log10(sensitivity / mm)

Adot = cedalion.data.get_precomputed_sensitivity(PROBE, HEAD_MODEL)

sensitivity = (
    Adot.sel(wavelength=SENS_WAVELENGTH, vertex=Adot.is_brain).sum("channel").values
)
log_sensitivity = np.log10(np.clip(sensitivity, 10.0 ** SENS_CLIM[0], None))

Adot

In [ ]:
if MAKE["sensitivity"]:
    plotter = new_plotter()
    vbx.plot_surface(
        plotter, head.brain, color=log_sensitivity, cmap="jet", clim=SENS_CLIM,
        smooth_shading=True,
        scalar_bar_args=dict(
            title=f"log10(sensitivity / mm)  @ {SENS_WAVELENGTH:.0f} nm",
            vertical=False, position_x=0.2, position_y=0.04, width=0.6, height=0.06,
            title_font_size=18, label_font_size=16, fmt="%.0f", n_labels=4,
        ),
    )
    plotter.add_text("NinjaNIRS 56x144 - whole-head sensitivity", font_size=12)

    frames = render_orbit(plotter, surface_center(head.brain), distance=420)
    fname = OUTPUT_DIR / "1_ninja_sensitivity.gif"
    save_gif(frames, fname)
    show_gif(fname)

## 2. NinjaNIRS probe on the head

`cedalion.data.get_ninja_cap_probe` returns template optode positions, landmarks and
the measurement list of the cap. To place the probe on this head, we use the same
registration as for the precomputed sensitivity: an initial affine alignment of the
landmarks, followed by `align_and_relax_to_scalp`. That second step pulls the
optodes onto the scalp while trying to preserve the source-detector distances of
the channels (see [51_spring_relaxation_registration.ipynb](51_spring_relaxation_registration.ipynb)).

Sources are drawn in red, detectors in blue, and channels as black lines. The scalp
is rendered at 30% opacity so that the brain stays visible.

In [ ]:
geo3d, landmarks, meas_list = cedalion.data.get_ninja_cap_probe()
geo3d = xr.concat((geo3d, landmarks), dim="label").pint.quantify("mm")

common_landmarks = geo3d.points.common_labels(head.landmarks)
align_mode = "general" if len(common_landmarks) > 3 else "trans_rot_isoscale"
geo3d_snapped, _ = head.align_and_relax_to_scalp(
    geo3d, meas_list, initial_align_mode=align_mode
)

optodes = geo3d_snapped[
    geo3d_snapped.type.isin([cdc.PointType.SOURCE, cdc.PointType.DETECTOR])
]
# one line per source-detector pair (the measurement list repeats wavelengths)
channels = meas_list[["source", "detector"]].drop_duplicates()
print(f"{optodes.sizes['label']} optodes, {len(channels)} channels")

In [ ]:
if MAKE["probe"]:
    plotter = new_plotter()
    vbx.plot_surface(plotter, head.brain, color="#d3a6a1", smooth_shading=True)
    vbx.plot_surface(plotter, head.scalp, color="w", opacity=0.3, smooth_shading=True)
    vbx.plot_labeled_points(plotter, optodes, meas_list=channels)
    plotter.add_text("NinjaNIRS 56x144 cap on Colin27", font_size=12)

    frames = render_orbit(plotter, surface_center(head.scalp), distance=520)
    fname = OUTPUT_DIR / "2_ninja_probe.gif"
    save_gif(frames, fname)
    show_gif(fname)

## 3. Schaefer parcellation on the pial and inflated surfaces

The head model files include an RGB colour for each Schaefer parcel. We map the
`parcel` label of each brain vertex to its colour, falling back to grey for vertices
without a label. The same colour array is used for the folded (pial) surface and for
the inflated surface. The inflated surface shows parcels hidden inside sulci and is
wider, so the camera is placed further away.

In [ ]:
if HEAD_MODEL == "colin27":
    hm_files = cedalion.data.get_colin27_headmodel_files()
else:
    hm_files = cedalion.data.get_icbm152_headmodel_files()
parcel_colors = hm_files.load_parcel_colors()

vertex_colors = np.array(
    [parcel_colors.get(p, [200, 200, 200]) for p in head.brain.vertices.parcel.values],
    dtype=np.uint8,
)
print(f"{len(set(head.brain.vertices.parcel.values))} parcels")

In [ ]:
for key, surface, distance, label in [
    ("parcels_pial", head.brain, 420, "pial"),
    ("parcels_inflated", infl, 620, "inflated"),
]:
    if not MAKE[key]:
        continue
    plotter = new_plotter()
    vbx.plot_surface(plotter, surface, color=vertex_colors, smooth_shading=True)
    plotter.add_text(f"Schaefer parcellation - {label} surface", font_size=12)

    frames = render_orbit(plotter, surface_center(surface), distance=distance)
    fname = OUTPUT_DIR / f"3{'a' if label == 'pial' else 'b'}_schaefer_{label}.gif"
    save_gif(frames, fname)
    show_gif(fname)

## 4. Block-averaged DOT image reconstruction

This section repeats the pipeline of
[40_image_reconstruction.ipynb](40_image_reconstruction.ipynb) in compact form:

1. Load the high-density finger-tapping recording and name its events.
2. Convert intensities to optical density, then correct motion artifacts (TDDR
   followed by wavelet filtering) and band-pass filter the data (0.01-0.5 Hz).
3. Cut epochs from -5 s to 30 s around each finger-tapping trial, subtract the
   pre-stimulus baseline, and average per condition.
4. Reconstruct HbO/HbR images on brain and scalp vertices from the block averages,
   using the precomputed sensitivity for this probe and head model.

Refer to that notebook for explanations of each step and the reconstruction
parameters.

In [ ]:
RECON_TRIAL = "FTapping/Right"
RECON_TIME_RANGE = (-5, 30, 0.5)  # start, stop, step in s

if MAKE["recon_multiview"] or MAKE["recon_rotating"]:
    rec = cedalion.data.get_fingertappingDOT()
    rec.stim.cd.rename_events({
        "1": "Control",
        "2": "FTapping/Left",
        "3": "FTapping/Right",
        "4": "BallSqueezing/Left",
        "5": "BallSqueezing/Right",
    })

    rec["od"] = cedalion.nirs.cw.int2od(rec["amp"])
    rec["od_tddr"] = motion.tddr(rec["od"])
    rec["od_wavelet"] = motion.wavelet(rec["od_tddr"])
    rec["od_freqfiltered"] = rec["od_wavelet"].cd.freq_filter(
        fmin=0.01, fmax=0.5, butter_order=4
    )

    epochs = rec["od_freqfiltered"].cd.to_epochs(
        rec.stim,
        ["FTapping/Left", "FTapping/Right"],
        before=5 * units.s,
        after=30 * units.s,
    )
    baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")
    blockaverage = (epochs - baseline).groupby("trial_type").mean("epoch")

    Adot_dot = cedalion.data.get_precomputed_sensitivity("fingertappingDOT", HEAD_MODEL)
    recon = dot.ImageRecon(
        Adot_dot,
        recon_mode="mua2conc",
        brain_only=False,
        alpha_meas=0.01,
        alpha_spatial=None,
        apply_c_meas=False,
        spatial_basis_functions=None,
    )
    img = recon.reconstruct(blockaverage)

    # the vis functions expect dims ("vertex", "chromo", "time")
    X_ts = img.sel(trial_type=RECON_TRIAL).rename({"reltime": "time"})
    X_ts = X_ts.transpose("vertex", "chromo", "time")

    # symmetric colour range from the 99th percentile of |HbO|
    scl = np.percentile(np.abs(X_ts.sel(chromo="HbO")).pint.dequantify(), 99)
    recon_clim = (-scl, scl)
    display(X_ts)

### 4a. Multi-view animation

`cedalion.vis.anatomy.image_recon_multi_view` renders five fixed views of the brain
and a colour bar, and writes one GIF frame per time step (here 71 frames from -5 s to
30 s). It sets frame timing through `fps`. We retime the result with `save_gif` so
that the whole time course plays in `DURATION_S`.

In [ ]:
if MAKE["recon_multiview"]:
    fname = OUTPUT_DIR / "4a_image_recon_multiview"
    cedalion.vis.anatomy.image_recon_multi_view(
        X_ts,
        head_ijk,
        cmap="seismic",
        clim=recon_clim,
        view_type="hbo_brain",
        title_str="HbO / µM",
        filename=str(fname),
        SAVE=True,
        time_range=RECON_TIME_RANGE * units.s,
        fps=6,
        geo3d_plot=None,
        wdw_size=(1024, 768),
    )
    gif = fname.with_suffix(".gif")
    save_gif(load_gif_frames(gif), gif)
    show_gif(gif)

### 4b. Rotating brain with the activation time course

Here, time and camera angle advance together: each of the `N_FRAMES` frames shows
the reconstructed HbO image at the nearest sample of an evenly spaced time grid,
while the camera completes one turn.

The update callback writes new values into the point data of the rendered mesh. This
only works if PyVista keeps the original vertices, so `split_sharp_edges=False` is
passed to `plot_surface`. The default splits vertices along sharp edges and changes
the vertex count.

Right-hand finger tapping activates the left motor cortex. The orbit therefore starts
at the right side (`theta0 = 0`), so the camera faces the left hemisphere halfway
through the loop, around the peak of the haemodynamic response. Change `theta0` to
move the viewpoint for other conditions.

In [ ]:
if MAKE["recon_rotating"]:
    X_hbo = X_ts.pint.dequantify().sel(chromo="HbO")
    X_hbo = X_hbo.sel(vertex=X_hbo.is_brain.values)
    frame_times = np.linspace(RECON_TIME_RANGE[0], RECON_TIME_RANGE[1], N_FRAMES)

    plotter = new_plotter()
    vbx.plot_surface(
        plotter, head.brain,
        color=X_hbo.sel(time=frame_times[0], method="nearest").values,
        cmap="seismic", clim=recon_clim,
        smooth_shading=True, split_sharp_edges=False,
        scalar_bar_args=dict(
            title="HbO / µM", vertical=False, position_x=0.2, position_y=0.04,
            width=0.6, height=0.06, title_font_size=18, label_font_size=16,
            fmt="%.1e", n_labels=3,
        ),
    )
    brain_mesh = list(plotter.actors.values())[0].mapper.dataset
    plotter.add_text(f"DOT image recon - {RECON_TRIAL} (block average)", font_size=12)
    time_label = plotter.add_text("", position="upper_right", font_size=12)

    def update(i_frame):
        t = frame_times[i_frame]
        brain_mesh.point_data["scalars"] = X_hbo.sel(time=t, method="nearest").values
        time_label.set_text("upper_right", f"t = {t:5.1f} s")

    frames = render_orbit(
        plotter, surface_center(head.brain), distance=420, theta0=0.0, update=update
    )
    fname = OUTPUT_DIR / "4b_image_recon_rotating.gif"
    save_gif(frames, fname)
    show_gif(fname)

## Adapting the animations

- **Other probes or head models:** change `PROBE` and `HEAD_MODEL`. The docstring of
  `cedalion.data.get_precomputed_sensitivity` lists the available combinations, e.g.
  `"ninja_uhd_cap_164x496"`. Section 2 uses the probe loader
  `cedalion.data.get_ninja_cap_probe` and must be switched to a matching loader.
- **Other per-vertex quantities:** anything with one value per brain vertex can be
  passed as `color=` to `vbx.plot_surface`. Examples are a single channel's
  sensitivity, GLM betas, or parcel-wise statistics mapped back to vertices.
- **Other colour maps and ranges:** adjust `cmap` and `clim`. For signed data such as
  concentration changes, keep the range symmetric so that zero maps to white in
  `"seismic"`.
- **Smaller files:** reduce `N_FRAMES` and `WINDOW_SIZE`. For video formats, pass the
  frames to `imageio` (`imageio.mimsave("out.mp4", frames, fps=...)`, requires
  `imageio-ffmpeg`).

In [ ]:
cedalion.bib.dump_to_notebook()